In [9]:
import pandas as pd

df = pd.read_csv("premier_league_raw.csv")
df[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]].head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,15/08/2025,Liverpool,Bournemouth,4,2,H
1,16/08/2025,Aston Villa,Newcastle,0,0,D
2,16/08/2025,Brighton,Fulham,1,1,D
3,16/08/2025,Sunderland,West Ham,3,0,H
4,16/08/2025,Tottenham,Burnley,3,0,H


In [10]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df = df.sort_values("Date").reset_index(drop=True)
df[["Date", "HomeTeam", "AwayTeam", "FTR"]].head()

,Date,HomeTeam,AwayTeam,FTR
0,2022-08-05,Crystal Palace,Arsenal,A
1,2022-08-06,Tottenham,Southampton,H
2,2022-08-06,Fulham,Liverpool,D
3,2022-08-06,Bournemouth,Aston Villa,H
4,2022-08-06,Leeds,Wolves,H


In [11]:
def get_points(row, team):
    if row["HomeTeam"] == team:
        if row["FTR"] == "H": return 3
        elif row["FTR"] == "D": return 1
        else: return 0
    elif row["AwayTeam"] == team:
        if row["FTR"] == "A": return 3
        elif row["FTR"] == "D": return 1
        else: return 0
    return None

In [ ]:
def calculate_form(df, n=5):
    home_form = []
    away_form = []
    
    for idx, row in df.iterrows():
        date = row["Date"]
        home_team = row["HomeTeam"]
        away_team = row["AwayTeam"]
        
        home_history = df[((df["HomeTeam"] == home_team) | (df["AwayTeam"] == home_team)) & (df["Date"] < date)].tail(n)
        away_history = df[((df["HomeTeam"] == away_team) | (df["AwayTeam"] == away_team)) & (df["Date"] < date)].tail(n)
        
        if len(home_history) > 0:
            home_points = home_history.apply(lambda r: get_points(r, home_team), axis=1)
            home_form.append(home_points.mean())
        else:
            home_form.append(None)
            
        if len(away_history) > 0:
            away_points = away_history.apply(lambda r: get_points(r, away_team), axis=1)
            away_form.append(away_points.mean())
        else:
            away_form.append(None)
    
    df["home_form"] = home_form
    df["away_form"] = away_form
    return df

df = calculate_form(df, n=5)
df[["Date", "HomeTeam", "AwayTeam", "home_form", "away_form", "FTR"]].tail(10)

,Date,HomeTeam,AwayTeam,home_form,away_form,FTR
1510,2026-05-24,Man City,Aston Villa,2.2,1.4,A
1511,2026-05-24,Nott'm Forest,Bournemouth,2.0,2.2,D
1512,2026-05-24,Liverpool,Brentford,1.4,1.0,D
1513,2026-05-24,Fulham,Newcastle,1.0,1.4,H
1514,2026-05-24,West Ham,Leeds,0.8,2.2,H
1515,2026-05-24,Burnley,Wolves,0.2,0.4,D
1516,2026-05-24,Brighton,Man United,1.4,2.6,A
1517,2026-05-24,Tottenham,Everton,1.6,0.4,H
1518,2026-05-24,Crystal Palace,Arsenal,0.4,2.4,A
1519,2026-05-24,Sunderland,Chelsea,1.0,0.8,H


In [13]:
def get_goal_diff(row, team):
    if row["HomeTeam"] == team:
        return row["FTHG"] - row["FTAG"]
    elif row["AwayTeam"] == team:
        return row["FTAG"] - row["FTHG"]
    return None

def calculate_goal_diff_trend(df, n=5):
    home_gd = []
    away_gd = []
    
    for idx, row in df.iterrows():
        date = row["Date"]
        home_team = row["HomeTeam"]
        away_team = row["AwayTeam"]
        
        home_history = df[((df["HomeTeam"] == home_team) | (df["AwayTeam"] == home_team)) & (df["Date"] < date)].tail(n)
        away_history = df[((df["HomeTeam"] == away_team) | (df["AwayTeam"] == away_team)) & (df["Date"] < date)].tail(n)
        
        if len(home_history) > 0:
            gd = home_history.apply(lambda r: get_goal_diff(r, home_team), axis=1)
            home_gd.append(gd.mean())
        else:
            home_gd.append(None)
            
        if len(away_history) > 0:
            gd = away_history.apply(lambda r: get_goal_diff(r, away_team), axis=1)
            away_gd.append(gd.mean())
        else:
            away_gd.append(None)
    
    df["home_goal_diff_trend"] = home_gd
    df["away_goal_diff_trend"] = away_gd
    return df

df = calculate_goal_diff_trend(df, n=5)
df[["Date", "HomeTeam", "AwayTeam", "home_goal_diff_trend", "away_goal_diff_trend"]].tail(10)

,Date,HomeTeam,AwayTeam,home_goal_diff_trend,away_goal_diff_trend
1510,2026-05-24,Man City,Aston Villa,1.4,0.2
1511,2026-05-24,Nott'm Forest,Bournemouth,1.8,1.0
1512,2026-05-24,Liverpool,Brentford,0.0,-0.2
1513,2026-05-24,Fulham,Newcastle,-0.6,0.4
1514,2026-05-24,West Ham,Leeds,-1.0,1.2
1515,2026-05-24,Burnley,Wolves,-1.4,-1.4
1516,2026-05-24,Brighton,Man United,0.6,0.8
1517,2026-05-24,Tottenham,Everton,0.2,-0.8
1518,2026-05-24,Crystal Palace,Arsenal,-1.6,1.0
1519,2026-05-24,Sunderland,Chelsea,-0.8,-1.0


In [ ]:
def calculate_h2h(df, n=5):
    h2h_home_points = []
    
    for idx, row in df.iterrows():
        date = row["Date"]
        home_team = row["HomeTeam"]
        away_team = row["AwayTeam"]
        
        h2h = df[
            (((df["HomeTeam"] == home_team) & (df["AwayTeam"] == away_team)) |
             ((df["HomeTeam"] == away_team) & (df["AwayTeam"] == home_team))) &
            (df["Date"] < date)
        ].tail(n)
        
        if len(h2h) > 0:
            points = h2h.apply(lambda r: get_points(r, home_team), axis=1)
            h2h_home_points.append(points.mean())
        else:
            h2h_home_points.append(None)
    
    df["h2h_home_points"] = h2h_home_points
    return df

df = calculate_h2h(df, n=5)
df[["Date", "HomeTeam", "AwayTeam", "h2h_home_points"]].tail(10)

,Date,HomeTeam,AwayTeam,h2h_home_points
1510,2026-05-24,Man City,Aston Villa,1.200000
1511,2026-05-24,Nott'm Forest,Bournemouth,0.400000
1512,2026-05-24,Liverpool,Brentford,2.400000
1513,2026-05-24,Fulham,Newcastle,1.200000
1514,2026-05-24,West Ham,Leeds,1.333333
1515,2026-05-24,Burnley,Wolves,1.333333
1516,2026-05-24,Brighton,Man United,1.800000
1517,2026-05-24,Tottenham,Everton,2.000000
1518,2026-05-24,Crystal Palace,Arsenal,0.200000
1519,2026-05-24,Sunderland,Chelsea,3.000000


In [15]:
feature_cols = ["home_form", "away_form", "home_goal_diff_trend", "away_goal_diff_trend", "h2h_home_points"]
print(df[feature_cols].isna().sum())
print(f"\nÖsszes sor: {len(df)}")

home_form                14
away_form                11
home_goal_diff_trend     14
away_goal_diff_trend     11
h2h_home_points         283
dtype: int64

Összes sor: 1520


In [16]:
df["h2h_home_points"] = df["h2h_home_points"].fillna(1)

before = len(df)
df = df.dropna(subset=["home_form", "away_form", "home_goal_diff_trend", "away_goal_diff_trend"])
after = len(df)

print(f"Kidobott sorok (nincs elég korábbi meccs): {before - after}")
print(f"Megmaradt sorok: {after}")


print(df[feature_cols].isna().sum())

Kidobott sorok (nincs elég korábbi meccs): 15
Megmaradt sorok: 1505
home_form               0
away_form               0
home_goal_diff_trend    0
away_goal_diff_trend    0
h2h_home_points         0
dtype: int64


In [17]:
target_map = {"H": 0, "D": 1, "A": 2}
df["target"] = df["FTR"].map(target_map)

df["target"].value_counts()

0    670
2    471
1    364
Name: target, dtype: int64

In [18]:
final_cols = ["Date", "HomeTeam", "AwayTeam", "home_form", "away_form", 
              "home_goal_diff_trend", "away_goal_diff_trend", "h2h_home_points", 
              "FTR", "target"]

df_final = df[final_cols].copy()
df_final.to_csv("premier_league_features.csv", index=False)

print(f"Végső feature tábla: {len(df_final)} sor, {len(df_final.columns)} oszlop")
df_final.head()

Végső feature tábla: 1505 sor, 10 oszlop


,Date,HomeTeam,AwayTeam,home_form,away_form,home_goal_diff_trend,away_goal_diff_trend,h2h_home_points,FTR,target
10,2022-08-13,Arsenal,Leicester,3.0,1.0,2.0,0.0,1.0,H,0
11,2022-08-13,Brighton,Newcastle,3.0,3.0,1.0,2.0,1.0,D,1
12,2022-08-13,Man City,Bournemouth,3.0,3.0,2.0,2.0,1.0,H,0
13,2022-08-13,Southampton,Leeds,0.0,3.0,-3.0,1.0,1.0,D,1
14,2022-08-13,Wolves,Fulham,0.0,1.0,-1.0,0.0,1.0,D,1


In [19]:
from azureml.core import Workspace

ws = Workspace.from_config()
datastore = ws.get_default_datastore()

datastore.upload_files(
    files=["premier_league_features.csv"],
    target_path="football-data/processed/",
    overwrite=True
)
print("Feltöltve a Blob Storage-ba")

"datastore.upload_files" is deprecated after version 1.0.69. Please use "FileDatasetFactory.upload_directory" instead. See Dataset API change notice at https://aka.ms/dataset-deprecation.


Uploading an estimated of 1 files
Uploading premier_league_features.csv
Uploaded premier_league_features.csv, 1 files out of an estimated total of 1
Uploaded 1 files
Feltöltve a Blob Storage-ba


In [20]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(DefaultAzureCredential())

data_asset = Data(
    path="azureml://datastores/workspaceblobstore/paths/football-data/processed/premier_league_features.csv",
    type=AssetTypes.URI_FILE,
    name="premier-league-features",
    version="1",
    description="Feldolgozott Premier League feature-ök: forma, gólkülönbség, H2H (2020-2026)"
)
ml_client.data.create_or_update(data_asset)
print("Feature Data Asset regisztrálva")

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Feature Data Asset regisztrálva
